# Prepare Geology (Plymouth)



## 1. Imports

In [1]:
#import libraries
import pandas as pd
import geopandas as gpd
import pyogrio
from pathlib import Path


import warnings 
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 120)

## 2. Configure Paths

In [2]:
PROJECT_DIR = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / "01_Data").exists()
)

In [3]:
# Define paths for raw data, boundary file, and output directory

RAW_DIGIMAP_DIR = PROJECT_DIR / "01_Data/Raw/BGS_Geology_Geothermal/Digimap_Downloads/Download_Plymouth_BGS_Geology_Groundwater_2999518"
BOUNDARY_GPKG = PROJECT_DIR / "01_Data/Processed/Boundaries/plymouth_boundaries.gpkg"
OUTPUT_DIR = PROJECT_DIR / "01_Data/Processed/Geothermal/model_outputs"
TABLE_OUTPUT_DIR = PROJECT_DIR / "03_Outputs/Tables"
TABLE_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEDROCK_SHEETS = {
    "ew348_plymouth": RAW_DIGIMAP_DIR / "bgs-50k_6431403" / "ew348" / "ew348_plymouth_bedrock.shp",
    "ew349_ivybridge": RAW_DIGIMAP_DIR / "bgs-50k_6431403" / "ew349" / "ew349_ivybridge_bedrock.shp",
}
TARGET_CRS = "EPSG:27700"

print("Raw Digimap data:    ", RAW_DIGIMAP_DIR.resolve())
print("Boundary GeoPackage: ", BOUNDARY_GPKG.resolve())
print("Output directory:    ", OUTPUT_DIR.resolve())
print("Raw folder exists:   ", RAW_DIGIMAP_DIR.exists())
print("Boundary file exists:", BOUNDARY_GPKG.exists())

Raw Digimap data:     /Users/kwakye/Desktop/Msc-Dissertation-Geothermal-Plymouth/01_Data/Raw/BGS_Geology_Geothermal/Digimap_Downloads/Download_Plymouth_BGS_Geology_Groundwater_2999518
Boundary GeoPackage:  /Users/kwakye/Desktop/Msc-Dissertation-Geothermal-Plymouth/01_Data/Processed/Boundaries/plymouth_boundaries.gpkg
Output directory:     /Users/kwakye/Desktop/Msc-Dissertation-Geothermal-Plymouth/01_Data/Processed/Geothermal/model_outputs
Raw folder exists:    True
Boundary file exists: True


## 3. Load Bedrock Sheets and LAD Boundary

In [4]:
ew348 = gpd.read_file(BEDROCK_SHEETS["ew348_plymouth"])
ew349 = gpd.read_file(BEDROCK_SHEETS["ew349_ivybridge"])
plymouth_lsoa = gpd.read_file(BOUNDARY_GPKG,layer="lsoa_plymouth_2021_clipped")

print("ew348 bedrock:", ew348.shape)
print("ew349 bedrock:", ew349.shape)
print("Plymouth LSOAs:", plymouth_lsoa.shape)

ew348 bedrock: (1761, 35)
ew349 bedrock: (413, 35)
Plymouth LSOAs: (164, 9)


## 4. Inspect Bedrock Sheets

In [5]:
print("ew348 crs:      ", ew348.crs)
print("ew349 crs:      ", ew349.crs)
print("Plymouth LSOA CRS:", plymouth_lsoa.crs)

ew348 crs:       EPSG:27700
ew349 crs:       EPSG:27700
Plymouth LSOA CRS: EPSG:27700


In [6]:
ew348[["LEX", "LEX_D", "RCS", "RCS_D"]].head()

,LEX,LEX_D,RCS,RCS_D
0,UIIDC,"UNNAMED IGNEOUS INTRUSION, DEVONIAN TO CARBONI...",MCGB,MICROGABBRO
1,UIIDC,"UNNAMED IGNEOUS INTRUSION, DEVONIAN TO CARBONI...",MCGB,MICROGABBRO
2,UIIDC,"UNNAMED IGNEOUS INTRUSION, DEVONIAN TO CARBONI...",MCGB,MICROGABBRO
3,UIIDC,"UNNAMED IGNEOUS INTRUSION, DEVONIAN TO CARBONI...",MCGB,MICROGABBRO
4,UIIDC,"UNNAMED IGNEOUS INTRUSION, DEVONIAN TO CARBONI...",MCGB,MICROGABBRO


In [7]:
# Check for missing data in the key lithology columns
print("ew348 missing values:\n", ew348[["LEX", "LEX_D", "RCS", "RCS_D"]].isna().sum())


ew348 missing values:
 LEX      0
LEX_D    0
RCS      0
RCS_D    0
dtype: int64


In [8]:
print("ew349 missing values:\n", ew349[["LEX", "LEX_D", "RCS", "RCS_D"]].isna().sum())

ew349 missing values:
 LEX      0
LEX_D    0
RCS      0
RCS_D    0
dtype: int64


In [9]:
# Check for invalid geometries
print("ew348 invalid geometries:      ", (~ew348.is_valid).sum())
print("ew349 invalid geometries:      ", (~ew349.is_valid).sum())
print("Plymouth LSOA invalid geometries:", (~plymouth_lsoa.is_valid).sum())

ew348 invalid geometries:       0
ew349 invalid geometries:       0
Plymouth LSOA invalid geometries: 0


In [10]:
# Check geometry types 
# several separate polygon pieces that share the same attributes (a multipart geometry)
print("ew348 geometry types:      ", ew348.geom_type.value_counts().to_dict())
print("ew349 geometry types:      ", ew349.geom_type.value_counts().to_dict())
print("Plymouth LSOA geometry types:",plymouth_lsoa.geom_type.value_counts().to_dict()
)

ew348 geometry types:       {'Polygon': 1761}
ew349 geometry types:       {'Polygon': 413}
Plymouth LSOA geometry types: {'MultiPolygon': 164}


## 5. Combine Bedrock Sheets and Reproject

In [11]:
ew348["source_sheet"] = "ew348_plymouth"
ew349["source_sheet"] = "ew349_ivybridge"

bedrock = gpd.GeoDataFrame(pd.concat(
        [ew348, ew349],
        ignore_index=True
    )
)

bedrock = bedrock.to_crs(TARGET_CRS)
plymouth_lsoa = plymouth_lsoa.to_crs(TARGET_CRS)

# Repair geometries
bedrock.geometry = bedrock.geometry.make_valid()
plymouth_lsoa.geometry = plymouth_lsoa.geometry.make_valid()

# Dissolve the clipped LSOAs to create bourndary for Plymouth
supply_boundary = gpd.GeoDataFrame(geometry=[plymouth_lsoa.geometry.union_all()],crs=TARGET_CRS)

supply_boundary.geometry = (supply_boundary.geometry.make_valid())

print("Combined bedrock:", bedrock.shape)
print("Combined bedrock CRS:", bedrock.crs)
print("Supply boundary area (km²):",supply_boundary.geometry.area.sum() / 1_000_000)

Combined bedrock: (2174, 36)
Combined bedrock CRS: EPSG:27700
Supply boundary area (km²): 79.7954836012946


## 6. Clip to Plymouth LAD Boundary

In [12]:
#clip bedrock to the plymouth boundary 
plymouth_bedrock = gpd.clip(bedrock, supply_boundary)

#splitting the multipolygons into single polygons
plymouth_bedrock = plymouth_bedrock.explode(index_parts=False).reset_index(drop=True)

In [13]:
print("Clipped bedrock polygons:", plymouth_bedrock.shape)
print("Invalid geometries after clip:", (~plymouth_bedrock.is_valid).sum())

Clipped bedrock polygons: (341, 36)
Invalid geometries after clip: 0


In [14]:
plymouth_bedrock.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 341 entries, 0 to 340
Data columns (total 36 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   LEX_WEB       341 non-null    object  
 1   LEX           341 non-null    object  
 2   LEX_D         341 non-null    object  
 3   LEX_RCS       341 non-null    object  
 4   RCS           341 non-null    object  
 5   RCS_X         341 non-null    object  
 6   RCS_D         341 non-null    object  
 7   RCS_ORIGIN    341 non-null    object  
 8   RANK          341 non-null    object  
 9   BED_EQ_D      341 non-null    object  
 10  MB_EQ_D       341 non-null    object  
 11  FM_EQ_D       341 non-null    object  
 12  SUBGP_EQ_D    341 non-null    object  
 13  GP_EQ_D       341 non-null    object  
 14  SUPGP_EQ_D    341 non-null    object  
 15  MAX_TIME_Y    341 non-null    float64 
 16  MIN_TIME_Y    341 non-null    float64 
 17  MAX_AGE       341 non-null    object  
 18  MA

In [15]:
plymouth_bedrock[["LEX", "LEX_D", "RCS", "RCS_D", "source_sheet"]].head()

,LEX,LEX_D,RCS,RCS_D,source_sheet
0,MDT,MEADFOOT GROUP,STSS,"SLATE, SILTSTONE AND SANDSTONE",ew349_ivybridge
1,MDT,MEADFOOT GROUP,STSS,"SLATE, SILTSTONE AND SANDSTONE",ew349_ivybridge
2,STG,STADDON FORMATION,SDSM,"SANDSTONE, SILTSTONE AND MUDSTONE",ew349_ivybridge
3,STG,STADDON FORMATION,SDSM,"SANDSTONE, SILTSTONE AND MUDSTONE",ew349_ivybridge
4,STG,STADDON FORMATION,SDSM,"SANDSTONE, SILTSTONE AND MUDSTONE",ew349_ivybridge


In [16]:
plymouth_bedrock[["LEX", "LEX_D", "RCS", "RCS_D", "source_sheet"]].nunique()

LEX             15
LEX_D           15
RCS             16
RCS_D           16
source_sheet     2
dtype: int64

In [17]:
c = plymouth_bedrock[plymouth_bedrock['LEX_D'] == 'SALTASH FORMATION']
c[c['RCS'] == 'CHRT']

,LEX_WEB,LEX,LEX_D,LEX_RCS,RCS,RCS_X,RCS_D,RCS_ORIGIN,RANK,BED_EQ_D,MB_EQ_D,FM_EQ_D,SUBGP_EQ_D,GP_EQ_D,SUPGP_EQ_D,MAX_TIME_Y,MIN_TIME_Y,MAX_AGE,MAX_EPOCH,MAX_SUBPER,MAX_PERIOD,MAX_ERA,MAX_EON,BGSTYPE,LEX_RCS_I,LEX_RCS_D,BGSREF,MAP_SRC,MAP_WEB,VERSION,RELEASED,NOM_SCALE,NOM_BGS_YR,UUID,source_sheet,geometry
323,http://www.bgs.ac.uk/Lexicon/lexicon.cfm?pub=SAH,SAH,SALTASH FORMATION,SAH-CHRT,CHRT,CHRT,CHERT,SEDIMENTARY,FORMATION,NOT APPLICABLE,NOT APPLICABLE,SALTASH FORMATION,NO PARENT,TAMAR GROUP,NO PARENT,407600000.0,346700000.0,EMSIAN,0.0,NOT DEFINED,DEVONIAN,PALAEOZOIC,PHANEROZOIC,BEDROCK,13303199_SAH-CHRT,SALTASH FORMATION - CHERT,900,EW348_PLYMOUTH,http://www.bgs.ac.uk/data/maps/maps.cfc?method...,8.24,28-07-2016,50000,1998,bgsn:DM50_V8_digmap1004081047200425,ew348_plymouth,"POLYGON ((247301 60268, 247254 60301, 247208 6..."


In [18]:
plymouth_bedrock["area_m2"] = plymouth_bedrock.geometry.area
plymouth_bedrock["area_km2"] = plymouth_bedrock["area_m2"] / 1_000_000
plymouth_bedrock = plymouth_bedrock[plymouth_bedrock["area_m2"] > 0].copy()

In [19]:
# Validate geology coverage against the dissolved
# clipped LSOA boundary

bedrock_union = (plymouth_bedrock.geometry.union_all())

boundary_union = (supply_boundary.geometry.union_all())

sum_polygon_area_km2 = (plymouth_bedrock.geometry.area.sum() / 1_000_000)

union_area_km2 = (bedrock_union.area/ 1_000_000)

boundary_area_km2 = (boundary_union.area/ 1_000_000)

overlap_area_km2 = (sum_polygon_area_km2- union_area_km2)

uncovered_area_km2 = (boundary_union.difference(bedrock_union).area / 1_000_000)

print("Sum of polygon areas:", sum_polygon_area_km2)
print("Unique geology coverage:", union_area_km2)
print("Supply boundary area:", boundary_area_km2)
print("Possible overlap area:", overlap_area_km2)
print("Uncovered supply-boundary area:", uncovered_area_km2)

Sum of polygon areas: 79.79548360129446
Unique geology coverage: 79.79548360129442
Supply boundary area: 79.79548360129434
Possible overlap area: 4.263256414560601e-14
Uncovered supply-boundary area: 0.0


In [20]:
#create lithology combinations 
plymouth_bedrock["model_unit_lithology_id"] = plymouth_bedrock["LEX"].astype(str) + "__" + plymouth_bedrock["RCS_D"].astype(str)
plymouth_bedrock['model_unit_lithology_id'].nunique()

32

In [21]:
plymouth_bedrock['model_unit_lithology_id'].unique()

array(['MDT__SLATE, SILTSTONE AND SANDSTONE',
       'STG__SANDSTONE, SILTSTONE AND MUDSTONE',
       'SAH__SLATE AND SILTSTONE',
       'LDEV__SANDSTONE AND [SUBEQUAL/SUBORDINATE] ARGILLACEOUS ROCKS, INTERBEDDED',
       'PYL__PYROCLASTIC-ROCK, BASALTIC', 'PYL__LAVA, BASALT',
       'PYL__LIMESTONE', 'PYL__HYALOCLASTITE', 'MDVL__LIMESTONE',
       'MDVS__PYROCLASTIC-ROCK, BASALTIC',
       'MDVL__PYROCLASTIC-ROCK, BASALTIC', 'MDVS__SLATE',
       'SAH__PYROCLASTIC-ROCK, BASALTIC', 'PRK__LIMESTONE',
       'FAR__LIMESTONE', 'SAH__HYALOCLASTITE',
       'TPT__MUDSTONE AND SILTSTONE', 'TPT__LIMESTONE',
       'TPT__HYALOCLASTITE', 'SAH__SLATE', 'SAH__LAVA, BASALTIC',
       'UDVS__PYROCLASTIC-ROCK, BASALTIC', 'MDVS__BASALTIC-ROCK',
       'WRG__SANDSTONE, SILTSTONE AND MUDSTONE', 'UDVS__BASALTIC-ROCK',
       'UIIDC__GABBRO', 'UIIDC__MICROGABBRO', 'SAH__LIMESTONE',
       'SAH__CHERT', 'UDP__FELSITE', 'UDVS__SLATE', 'TVY__SLATE'],
      dtype=object)

In [22]:
plymouth_bedrock = plymouth_bedrock.sort_values(
    ["source_sheet", "LEX", "RCS_D", "area_km2"], ascending=[True, True, True, False]
).reset_index(drop=True)

In [23]:
plymouth_bedrock[["source_sheet", "LEX", "LEX_D", "RCS", "RCS_D", "area_km2"]].head()

,source_sheet,LEX,LEX_D,RCS,RCS_D,area_km2
0,ew348_plymouth,FAR,FARADAY ROAD MEMBER,LMST,LIMESTONE,0.068762
1,ew348_plymouth,FAR,FARADAY ROAD MEMBER,LMST,LIMESTONE,0.042912
2,ew348_plymouth,FAR,FARADAY ROAD MEMBER,LMST,LIMESTONE,0.042779
3,ew348_plymouth,FAR,FARADAY ROAD MEMBER,LMST,LIMESTONE,0.024289
4,ew348_plymouth,FAR,FARADAY ROAD MEMBER,LMST,LIMESTONE,0.022360


## 6. Create thermal properties mapping based on lithology

In [24]:
thermal_properties_csv = PROJECT_DIR /'01_Data/Processed/Geothermal/Plymouth_Thermal_Properties_Final.csv'
    
thermal_properties = pd.read_csv(thermal_properties_csv, encoding="utf-8-sig").copy()
thermal_properties.head()

,engineering_class_id,engineering_geology_class,mapped_lithology,thermal_conductivity_w_mk,lambda_low_w_mk,lambda_high_w_mk,density_kg_m3,specific_heat_capacity_j_kgk,volumetric_heat_capacity_mj_m3k,alpha_low_m2_s,thermal_diffusivity_m2_s,alpha_high_m2_s,evidence_type,evidence_confidence,primary_source,source_url,selection_note
0,5,basaltic lava/volcanic rock,BASALTIC-ROCK,1.8,1.6,2.30,2750,950,2.612,6.124000e-07,6.890000e-07,8.804000e-07,near-exact BGS/UK measured baseline,Moderate,BGS OR/25/014 Appendix 1: Devonian lava/extrus...,https://nora.nerc.ac.uk/id/eprint/539481/1/BGS...,Use as the baseline for undifferentiated basal...
1,7,chert,CHERT,3.2,2.8,3.50,2670,880,2.350,1.192000e-06,1.362000e-06,1.490000e-06,BGS chert analogue; representative interpreted,Moderate-Low,BGS OR/25/014 Appendix 1: Jurassic chert 2.80 ...,https://nora.nerc.ac.uk/id/eprint/539481/1/BGS...,Representative is a rounded value within the B...
2,6,intrusive igneous rock,FELSITE,3.0,2.2,3.20,2650,840,2.226,9.883000e-07,1.348000e-06,1.438000e-06,BGS felsic-rock proxy,Low,BGS OR/25/014 Appendix 1: fine-grained acid ro...,https://nora.nerc.ac.uk/id/eprint/539481/1/BGS...,Use provisionally; Plymouth felsite may includ...
3,6,intrusive igneous rock,GABBRO,2.4,2.4,3.00,2930,880,2.578,9.308000e-07,9.308000e-07,1.164000e-06,direct international gabbro measurement plus B...,Moderate-Low,"Hyndman and Drury, DSDP Leg 37: gabbro 2.40 W/...",https://deepseadrilling.org/37/Volume/dsdp37_1...,Use 2.40 as the direct same-lithology measurem...
4,4,volcaniclastic/hyaloclastite,HYALOCLASTITE,2.6,2.5,2.75,2550,900,2.295,1.089000e-06,1.133000e-06,1.198000e-06,international saturated direct-lithology analogue,Low,Scott et al. 2023 Valgardur database: two wate...,https://essd.copernicus.org/articles/15/1165/2...,Representative is the rounded midpoint of the ...


In [25]:
#create a join key for lithology matching
thermal_properties["lithology_join_key"] = (
    thermal_properties["mapped_lithology"]
    .astype("string").str.normalize("NFKC")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)

plymouth_bedrock["lithology_join_key"] = (
    plymouth_bedrock["RCS_D"]
    .astype("string")
    .str.normalize("NFKC")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)

In [26]:
bedrock_lithologies = sorted(plymouth_bedrock["lithology_join_key"].dropna().unique())
thermal_lithologies = sorted(thermal_properties["lithology_join_key"].dropna().unique())

missing_thermal_properties = sorted(set(bedrock_lithologies) - set(thermal_lithologies))
unused_thermal_properties = sorted(set(thermal_lithologies) - set(bedrock_lithologies))

print("Distinct Plymouth bedrock lithologies:", len(bedrock_lithologies))
print("Distinct thermal property lithologies:", len(thermal_lithologies))
print("Missing thermal property lithologies:", missing_thermal_properties)
print("Thermal lithologies not present in Plymouth bedrock:", unused_thermal_properties)

Distinct Plymouth bedrock lithologies: 16
Distinct thermal property lithologies: 16
Missing thermal property lithologies: []
Thermal lithologies not present in Plymouth bedrock: []


In [27]:
thermal_join_columns = [
    "lithology_join_key",
    "thermal_conductivity_w_mk",
    "lambda_low_w_mk",
    "lambda_high_w_mk",
    "density_kg_m3",
    "specific_heat_capacity_j_kgk",
    "volumetric_heat_capacity_mj_m3k",
    "alpha_low_m2_s",
    "thermal_diffusivity_m2_s",
    "alpha_high_m2_s",
]

thermal_output_columns = [col for col in thermal_join_columns if col != "lithology_join_key"]
existing_thermal_columns = [col for col in thermal_output_columns if col in plymouth_bedrock.columns]

plymouth_bedrock = plymouth_bedrock.drop(columns=existing_thermal_columns)

rows_before_thermal_join = len(plymouth_bedrock)

plymouth_bedrock = plymouth_bedrock.merge(thermal_properties[thermal_join_columns], on="lithology_join_key",how="left")

In [28]:
plymouth_bedrock[thermal_join_columns].head()

,lithology_join_key,thermal_conductivity_w_mk,lambda_low_w_mk,lambda_high_w_mk,density_kg_m3,specific_heat_capacity_j_kgk,volumetric_heat_capacity_mj_m3k,alpha_low_m2_s,thermal_diffusivity_m2_s,alpha_high_m2_s
0,LIMESTONE,3.0,2.7,3.2,2680,890,2.385,0.000001,0.000001,0.000001
1,LIMESTONE,3.0,2.7,3.2,2680,890,2.385,0.000001,0.000001,0.000001
2,LIMESTONE,3.0,2.7,3.2,2680,890,2.385,0.000001,0.000001,0.000001
3,LIMESTONE,3.0,2.7,3.2,2680,890,2.385,0.000001,0.000001,0.000001
4,LIMESTONE,3.0,2.7,3.2,2680,890,2.385,0.000001,0.000001,0.000001


## 7. Summarise Geological Units

In [29]:
unit_summary = (
    plymouth_bedrock
    .groupby([ "LEX_D", "RCS_D", "model_unit_lithology_id"], dropna=False)
    .agg(polygon_count=("geometry", "size"),area_km2=("area_km2", "sum"),)
    .reset_index()
    .sort_values("area_km2", ascending=False)
)
unit_summary["area_percent_of_plymouth_bedrock"] = unit_summary["area_km2"] / unit_summary["area_km2"].sum() * 100

unit_summary.head(10)

,LEX_D,RCS_D,model_unit_lithology_id,polygon_count,area_km2,area_percent_of_plymouth_bedrock
30,UPPER DEVONIAN SLATES,SLATE,UDVS__SLATE,3,25.880615,32.433684
19,SALTASH FORMATION,SLATE AND SILTSTONE,SAH__SLATE AND SILTSTONE,69,17.934724,22.475864
24,TORPOINT FORMATION,MUDSTONE AND SILTSTONE,TPT__MUDSTONE AND SILTSTONE,54,11.636419,14.582804
7,MIDDLE DEVONIAN SLATES,SLATE,MDVS__SLATE,7,7.554552,9.467393
3,MIDDLE DEVONIAN LIMESTONE,LIMESTONE,MDVL__LIMESTONE,7,5.249781,6.579045
21,TAVY FORMATION,SLATE,TVY__SLATE,1,3.337381,4.182419
10,PLYMOUTH LIMESTONE FORMATION,LIMESTONE,PYL__LIMESTONE,57,1.457918,1.827068
6,MIDDLE DEVONIAN SLATES,"PYROCLASTIC-ROCK, BASALTIC","MDVS__PYROCLASTIC-ROCK, BASALTIC",7,1.219067,1.527739
15,SALTASH FORMATION,"LAVA, BASALTIC","SAH__LAVA, BASALTIC",33,1.167971,1.463706
29,UPPER DEVONIAN SLATES,"PYROCLASTIC-ROCK, BASALTIC","UDVS__PYROCLASTIC-ROCK, BASALTIC",15,1.016773,1.274224


## 9. Save Outputs

In [30]:
output_gpkg = OUTPUT_DIR / "01_prepared_geology.gpkg"
summary_csv = OUTPUT_DIR / "01_prepared_geology_unit_summary.csv"

if output_gpkg.exists():
    output_gpkg.unlink()

plymouth_bedrock.to_file(output_gpkg, layer="prepared_geology", driver="GPKG")
unit_summary.to_csv(summary_csv, index=False)

In [31]:
#check saved layers
pyogrio.list_layers(output_gpkg)

array([['prepared_geology', 'Polygon']], dtype=object)